# XP Ninja: Build an Intelligent Document Assistant
Author: arielzin33@gmail.com

An LLM-powered assistant that summarizes a contract, answers follow-up questions as a role-based contract lawyer, maintains context across turns via memory chaining, and critiques its own answers.

Note: the shared Colab template requires Google sign-in and could not be fetched automatically, so this notebook reproduces the four-step workflow from the written instructions, using the provided Service Agreement excerpt as the working document. Copy/merge into the shared template as needed.

## Input Document (Service Agreement Excerpt)

```
Service Agreement – Excerpt
This Service Agreement ("Agreement") is made effective as of March 1, 2025, by and between BrightLine Technologies Ltd., hereinafter referred to as "Provider", and NovaWare Systems Inc., hereinafter referred to as "Client".

1. Scope of Work: Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Exhibit A.
2. Payment Terms: Client agrees to pay a fixed monthly fee of $12,000, payable within 30 days of receipt of invoice. Late payments will incur a 2% penalty per month.
3. Term and Termination: This Agreement shall commence on March 1, 2025, and remain in effect for 12 months. Either party may terminate with 30 days' written notice.
4. Confidentiality: Both parties agree to protect the confidentiality of proprietary or sensitive information shared during the course of the engagement.
5. Limitation of Liability: Provider's total liability shall not exceed the fees paid by Client in the 3 months prior to a claim. Provider is not liable for indirect or consequential damages.
6. Governing Law: This Agreement shall be governed by the laws of the State of California.
```

---
## 🧩 Step 1: Initial Summary Prompt

Chain-of-Thought prompt that walks through the document section by section before producing a plain-English summary, ensuring the four required elements (responsibilities, payment, termination, liability) are all captured accurately rather than skipped or blended.

### Prompt

> You are summarizing a legal service agreement for a non-lawyer business audience. Read the document below and **think through it section by section before writing your summary**:
>
> Step 1: Identify what the Provider (BrightLine Technologies Ltd.) is responsible for.
> Step 2: Identify what the Client (NovaWare Systems Inc.) is responsible for, especially payment obligations and deadlines.
> Step 3: Identify how and when the agreement can end.
> Step 4: Identify the limits on the Provider's liability.
> Step 5: Combine these into a single plain-English summary of **4 short bullet points** — one each for: (a) key responsibilities of each party, (b) payment terms, (c) termination clauses, (d) liability limitations. Avoid legal jargon; write as if explaining to a busy startup founder.
>
> Document: "{{document_text}}"
>
> Show your step-by-step reasoning first, then give the final 4-bullet summary labeled **"Final Summary:"**.

### Example expected output

**Final Summary:**
- **Responsibilities:** BrightLine (Provider) manages NovaWare's (Client) cloud infrastructure — monitoring, incident response, and monthly reports (details in Exhibit A); NovaWare's main obligation is to pay on time.
- **Payment:** NovaWare pays a flat $12,000/month, due within 30 days of invoice; late payments add a 2% monthly penalty.
- **Termination:** The agreement runs for 12 months starting March 1, 2025; either side can end it early with 30 days' written notice.
- **Liability:** BrightLine's total liability is capped at whatever NovaWare paid in the last 3 months, and BrightLine isn't liable for indirect or consequential damages.

---
## 💬 Step 2: Role-Based Follow-Up Q&A

Role prompt casting the assistant as a contract lawyer, using **Instance-Adaptive CoT** — the depth of reasoning adapts to how complex the specific question is — combined with a **Narrative-of-Thought** delivery style so answers feel like a lawyer explaining the clause conversationally, not a dry recital.

### Prompt

> **Act as an experienced contract lawyer** advising a client (in plain, friendly language, not legalese) about a Service Agreement between BrightLine Technologies Ltd. (Provider) and NovaWare Systems Inc. (Client). You have already reviewed the full agreement, reproduced below for reference.
>
> When answering a question:
> - **Adapt your depth of reasoning to the question's complexity** — a simple factual question (e.g., "when does this end?") gets a short direct answer; a nuanced question (e.g., "what's my actual financial exposure if BrightLine causes an outage?") gets a brief walk-through of the relevant clause's reasoning before the conclusion.
> - **Explain like you're talking to the client across a table**, not reading a statute — use a short narrative framing ("Here's what this clause actually means for you...") rather than just quoting contract text.
> - Always ground your answer in the specific clause number(s) from the agreement.
>
> Agreement: "{{document_text}}"
>
> Client question: "Can you explain the limitation of liability clause?"

### Example expected output

> Sure — let's break down Clause 5. Basically, if something goes wrong on BrightLine's end and you need to make a claim, the most you could recover is capped at whatever you've paid them over the **previous 3 months**. So if you're paying $12,000/month, your maximum recovery would be around $36,000, regardless of how big the actual damage was.
>
> On top of that, BrightLine isn't on the hook for *indirect or consequential* damages — think lost profits, reputational harm, that kind of downstream impact. Only direct losses, up to that 3-month cap, are recoverable. It's a fairly standard liability cap for service agreements, but worth knowing going in, especially since your infrastructure is business-critical.

---
## 🔁 Step 3: Memory Integration

**Method chosen: Structured conversation history.** Rather than replaying the entire raw chat log (prior message passing) or standing up a vector store for a single short document, a compact structured object holds the document summary plus a running log of Q&A already covered. This is injected into every new prompt so the assistant never re-summarizes from scratch or contradicts an earlier answer.

### Structured context object (after Steps 1 & 2)

```json
{
  "document_summary": {
    "responsibilities": "BrightLine manages NovaWares cloud infrastructure (monitoring, incident response, monthly reporting per Exhibit A); NovaWares core duty is timely payment.",
    "payment_terms": "$12,000/month flat fee, due within 30 days of invoice; 2%/month late penalty.",
    "termination": "12-month term from March 1, 2025; either party may terminate with 30 days written notice.",
    "liability": "Providers liability capped at fees paid in the prior 3 months; no liability for indirect/consequential damages."
  },
  "qa_history": [
    {
      "question": "Can you explain the limitation of liability clause?",
      "answer_summary": "Explained the 3-month fee cap and exclusion of indirect/consequential damages under Clause 5."
    }
  ]
}
```

### New prompt using this context

> You are the contract lawyer assistant reviewing the Service Agreement between BrightLine Technologies Ltd. and NovaWare Systems Inc. Here is what has already been established in this conversation:
>
> **Document summary:** {{document_summary}}
>
> **Previously answered:** {{qa_history}}
>
> The client now asks: "If I want to end this early, does that affect the liability cap you mentioned earlier?"
>
> Answer using the context above — **do not re-summarize the whole contract**, and explicitly connect this answer to the liability explanation already given if relevant.

---
## 🔍 Step 4: Mitigation & Refinement

Self-reflection prompt that has the assistant critique its own prior answer for accuracy (grounded strictly in the contract text) and clarity, plus an optional second-agent legal-review pass to catch misinterpretations before the answer reaches the client.

### Self-reflection prompt

> Review the answer you just gave to the client's question, reproduced below along with the original contract text. Critique your own answer against these criteria:
>
> 1. **Accuracy:** Does every claim in the answer trace back to specific language in the contract? Flag anything that was inferred, assumed, or not explicitly stated.
> 2. **Completeness:** Did the answer address the full question, including any related clause the client should be aware of (e.g., interactions between termination and liability)?
> 3. **Clarity:** Is the explanation understandable to a non-lawyer, free of unnecessary legal jargon?
>
> If you find an issue, **rewrite the corrected answer**. If the original answer holds up, state "No corrections needed" and explain briefly why.
>
> Contract: "{{document_text}}"
> Your prior answer: "{{previous_answer}}"

### Optional: Multi-agent critique step

> **Agent 1 (Drafting Lawyer):** produces the initial answer to the client's question, as in Step 2.
>
> **Agent 2 (Reviewing Lawyer — adversarial critique role):** "You are a second, more skeptical contract lawyer reviewing a colleague's answer before it goes to the client. Read the contract and the colleague's answer below. Identify any statement that is not directly supported by the contract text, any missing caveat a client should know, or any place the answer could be misread. List issues found, or state 'No issues found.' Do not rewrite the answer yourself — only flag problems for the drafting lawyer to fix."
>
> **Agent 1 (revision pass):** receives Agent 2's flagged issues and produces a final corrected answer, incorporating the fixes.
>
This adversarial two-agent pattern (drafter → critic → reviser) catches hallucinated or overstated legal claims before they reach the end user — useful precisely because legal misinterpretation carries real consequences, and a single model instance is prone to confidently repeating its own errors rather than catching them unprompted.